# Build income-group-differentiated CE parameters for 142 economies

**Why this notebook.** The circular-economy (CE) scenario originally applied a single global 2050 target to every economy (e.g. 20% recycling, 25% clinker reduction). Reviewers asked (i) where these numbers come from and (ii) whether Global-South economies can realistically match developed-country rates.

**Fix.** We differentiate every CE lever by **World Bank income group** (HIC / UMIC / LMIC / LIC). Each 2050 target is **anchored to the governing concrete standard or currently observed best-practice within that income group**, and set at or below demonstrated feasibility (conservative). High-income groups get higher targets; low-income groups lower — directly reflecting differential infrastructure and institutional capacity.

## CE levers and income-group 2050 targets (conservative)

| Lever | LIC | LMIC | UMIC | HIC | Anchor / source |
|---|---|---|---|---|---|
| End-of-life recycling (agg.) | 5% | 10% | 15% | 20% | **HIC 20% = modal RCA-in-structural-concrete cap** (BS 8500, Brazil NBR 15116, India IS 383, ES/NL/HK all 20%); also the mass-balance ceiling (EoL coarse outflow ≤ concurrent inflow while stock grows). Lower groups scaled by circular-infrastructure capacity — Global-South C&D sorting / RCA-certification / market gaps; <10% recycling observed in CN/IN, closed-loop <5% even in the EU today |
| Sand substitution (ore-sand) | 5% | 10% | 15% | 20% | Demand-side acceptance ceiling for ore-sand (fine aggregate); within the 10–40% technical band (Golev 2022 p.61); supply bounded in 5-estimate-oresand-supply-cap |
| Clinker-to-cement ratio (2050 target) | 0.68 | 0.64 | 0.60 | 0.56 | Absolute target k-factor (not a reduction): HIC 0.56 ≈ IEA NZE 2050 (~0.57); GCCA 2050 Roadmap; LC3 (~0.50 feasible); ladder = differential SCM/LC3 access |
| Component reuse | 2% | 5% | 8% | 10% | Hopkinson 2019; conservative across all groups |
| Lifetime extension | +20% | +30% | +40% | +50% | Cai 2015 (short-lived CN buildings); Miller 2020; O'Connor 2004 |
| More intensive use (per-cap stock ↓) | 2% | 3% | 4% | 5% | Pauliuk 2021; Song 2023 (Nat. Clim. Chang.) |

All targets are **2050 end-points**; the model ramps each linearly from 0 (or current) in 2022 to the target in 2050 (per SI Eqs. S47–S55).

## Output
`data/CE-parameters-142.csv` — one row per economy, columns = the six 2050 targets + income group + source tag.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'

## Step 1 — World Bank income-group mapping (ISO3 → HIC/UMIC/LMIC/LIC)

Based on the World Bank FY2024 income classification (GNI per capita, Atlas method). Historical/aggregate non-standard codes in the 142 list (SUN=former USSR, YUG=former Yugoslavia, CSK=former Czechoslovakia, EWP, RNF) and overseas territories are assigned to the income group of their present-day successor / parent economy, or to a sensible default; these have negligible future demand in the model. Taiwan (TWN) = HIC; Hong Kong (HKG), Macau (MAC), Singapore (SGP) = HIC.

In [3]:
# World Bank FY2024 income groups for the 142 economies.
INCOME_GROUP = {
    # --- High income (HIC) ---
    'ARE':'HIC','AUS':'HIC','AUT':'HIC','BEL':'HIC','BHR':'HIC','BHS':'HIC','BRB':'HIC','BRN':'HIC',
    'CAN':'HIC','CHE':'HIC','CHL':'HIC','CYP':'HIC','DEU':'HIC','DNK':'HIC','ESP':'HIC','FIN':'HIC',
    'FRA':'HIC','GBR':'HIC','GRC':'HIC','HKG':'HIC','HUN':'HIC','IRL':'HIC',
    'ISL':'HIC','ISR':'HIC','ITA':'HIC','JPN':'HIC','KOR':'HIC','KWT':'HIC','MAC':'HIC',
    'NLD':'HIC','NOR':'HIC','NZL':'HIC','OMN':'HIC','PAN':'HIC','POL':'HIC','PRT':'HIC',
    'QAT':'HIC','SAU':'HIC','SGP':'HIC','SWE':'HIC','TTO':'HIC','TWN':'HIC','URY':'HIC','USA':'HIC',
    'GUF':'HIC','GLP':'HIC','MTQ':'HIC','REU':'HIC','NCL':'HIC',          # French overseas territories
    'CSK':'HIC',                                                          # former Czechoslovakia -> CZE/SVK now HIC
    # --- Upper-middle income (UMIC) ---
    'ALB':'UMIC','ARG':'UMIC','BGR':'UMIC','BRA':'UMIC','BWA':'UMIC','CHN':'UMIC','COL':'UMIC',
    'CRI':'UMIC','CUB':'UMIC','DOM':'UMIC','DZA':'UMIC','ECU':'UMIC','FJI':'UMIC','GAB':'UMIC',
    'GUY':'UMIC','IRN':'UMIC','IRQ':'UMIC','JAM':'UMIC','LBY':'UMIC','MEX':'UMIC','MYS':'UMIC',
    'NAM':'UMIC','PER':'UMIC','PRY':'UMIC','ROU':'UMIC','SUR':'UMIC','THA':'UMIC',
    'TUN':'UMIC','TUR':'UMIC','VEN':'UMIC','ZAF':'UMIC',
    'SUN':'UMIC','YUG':'UMIC','RNF':'UMIC','EWP':'UMIC',                  # aggregates -> middle default
    # --- Lower-middle income (LMIC) ---
    'AGO':'LMIC','BEN':'LMIC','BOL':'LMIC','BTN':'LMIC','CIV':'LMIC','CMR':'LMIC','COG':'LMIC',
    'CPV':'LMIC','DJI':'LMIC','EGY':'LMIC','GHA':'LMIC','GTM':'LMIC','HND':'LMIC','IDN':'LMIC',
    'IND':'LMIC','JOR':'LMIC','KEN':'LMIC','KHM':'LMIC','LAO':'LMIC','LBN':'LMIC','LKA':'LMIC',
    'MAR':'LMIC','MMR':'LMIC','MNG':'LMIC','MRT':'LMIC','NGA':'LMIC','NIC':'LMIC','NPL':'LMIC',
    'PHL':'LMIC','PNG':'LMIC','SEN':'LMIC','SLV':'LMIC','TZA':'LMIC','VNM':'LMIC',
    'PRK':'LMIC',                                                         # DPRK: WB unclassified -> LMIC proxy
    # --- Low income (LIC) ---
    'AFG':'LIC','BDI':'LIC','BFA':'LIC','TCD':'LIC','COD':'LIC','ERI':'LIC','ETH':'LIC',
    'GIN':'LIC','HTI':'LIC','LBR':'LIC','MDG':'LIC','MLI':'LIC','MOZ':'LIC','NER':'LIC',
    'RWA':'LIC','SDN':'LIC','SLE':'LIC','SOM':'LIC','SYR':'LIC','TGO':'LIC','UGA':'LIC','YEM':'LIC',
}
print('mapped:', len(INCOME_GROUP))

mapped: 142


In [4]:
# Verify every economy in the 142 list has an income group.
econ = pd.read_csv(RESULTS / 'pseudo-codes-economies-142.csv')
isos = sorted(econ['economies_142'].dropna().unique().tolist())
missing = [c for c in isos if c not in INCOME_GROUP]
print(f'economies: {len(isos)} | missing income group: {missing}')
assert not missing, f'Add income groups for: {missing}'

from collections import Counter
print('group counts:', Counter(INCOME_GROUP[c] for c in isos))

economies: 142 | missing income group: []
group counts: Counter({'HIC': 50, 'LMIC': 35, 'UMIC': 35, 'LIC': 22})


## Step 2 — Income-group 2050 targets per CE lever (conservative)

Each value is the 2050 end-point; the MFA model ramps it linearly from 2022 (SI Eqs. S47–S55).

In [ ]:
# rows = income group, cols = CE lever. Fractions (0-1) except lifetime_ext (fractional increase) and clinker_ratio_target (absolute 2050 clinker-to-cement ratio).
CE_TARGETS = pd.DataFrame({
    'recycling'      : {'LIC':0.05, 'LMIC':0.10, 'UMIC':0.15, 'HIC':0.20},
    'sand_substitution':{'LIC':0.05, 'LMIC':0.10, 'UMIC':0.15, 'HIC':0.20},
    'clinker_ratio_target':{'LIC':0.68, 'LMIC':0.64, 'UMIC':0.60, 'HIC':0.56},
    'reuse'          : {'LIC':0.02, 'LMIC':0.05, 'UMIC':0.08, 'HIC':0.10},
    'lifetime_ext'   : {'LIC':0.20, 'LMIC':0.30, 'UMIC':0.40, 'HIC':0.50},
    'intensive_use'  : {'LIC':0.02, 'LMIC':0.03, 'UMIC':0.04, 'HIC':0.05},
})
# source tag per lever (for the SI / documentation)
CE_SOURCE = {
    'recycling'        : 'HIC 20% = modal RCA structural-concrete cap (BS 8500/NBR 15116/IS 383); lower income groups scaled by C&D recycling-infrastructure capacity',
    'sand_substitution' : 'sand substitution by ore-sand (fine aggregate only); demand-side acceptance ceiling, HIC 20% within the 10-40% technical band (Golev 2022 p.61); supply bounded separately in 5-estimate-oresand-supply-cap (Golev 2022 p.74 fn19 & p.82; Franks 2025)',
    'clinker_ratio_target': 'absolute 2050 clinker-to-cement ratio target (k-factor), not a reduction: HIC 0.56 ~ IEA NZE 2050 (~0.57); GCCA 2050 Roadmap; LC3 ~0.50 feasible; income-group ladder reflects differential SCM/LC3 availability',
    'reuse'            : 'Gobbo 2024 (struct 1-5%, skin 5-15% by mass); Kupfer 2023 (77 concrete-reuse precedents); Hopkinson 2019; conservative vs Pauliuk 2021',
    'lifetime_ext'     : 'Cai 2015; Miller 2020; O’Connor 2004',
    'intensive_use'    : 'Pauliuk 2021; Song 2023',
}
CE_TARGETS

## Step 3 — Assemble the 142-economy parameter table and save

In [6]:
rows = []
for iso in isos:
    g = INCOME_GROUP[iso]
    row = {'economy': iso, 'income_group': g}
    for lever in CE_TARGETS.columns:
        row[lever] = float(CE_TARGETS.loc[g, lever])
    rows.append(row)

ce_params = pd.DataFrame(rows).set_index('economy')
out = DATA / 'CE-parameters-142.csv'
ce_params.to_csv(out)
print(f'Saved {out}  ({ce_params.shape[0]} economies x {ce_params.shape[1]} cols)')
ce_params.head(10)

Saved d:\wty\Documents\GitHub\aggregate-mining-biodiversity\data\CE-parameters-142.csv  (142 economies x 7 cols)


,income_group,recycling,agg_substitution,clinker_reduction,reuse,lifetime_ext,intensive_use
economy,,,,,,,
AFG,LIC,0.05,0.05,0.10,0.02,0.2,0.02
AGO,LMIC,0.10,0.10,0.15,0.05,0.3,0.03
ALB,UMIC,0.20,0.15,0.20,0.08,0.4,0.04
ARE,HIC,0.35,0.25,0.30,0.12,0.5,0.05
ARG,UMIC,0.20,0.15,0.20,0.08,0.4,0.04
AUS,HIC,0.35,0.25,0.30,0.12,0.5,0.05
AUT,HIC,0.35,0.25,0.30,0.12,0.5,0.05
BDI,LIC,0.05,0.05,0.10,0.02,0.2,0.02
BEL,HIC,0.35,0.25,0.30,0.12,0.5,0.05


In [ ]:
# Sanity: show the per-group values and a few well-known economies.
print('=== distinct rows by income group ===')
print(ce_params.drop_duplicates('income_group').sort_values('income_group').to_string())
print()
print('=== spot check ===')
for iso in ['USA','DEU','CHN','IND','IDN','NGA','ETH','SAU']:
    if iso in ce_params.index:
        r = ce_params.loc[iso]
        print(f"  {iso} ({r['income_group']}): recyc={r['recycling']:.2f} sub={r['sand_substitution']:.2f} "
              f"clinker_k={r['clinker_ratio_target']:.2f} life=+{r['lifetime_ext']:.0%}")

=== distinct rows by income group ===
        income_group  recycling  agg_substitution  clinker_reduction  reuse  lifetime_ext  intensive_use
economy                                                                                                 
ARE              HIC       0.35              0.25               0.30   0.12           0.5           0.05
AFG              LIC       0.05              0.05               0.10   0.02           0.2           0.02
AGO             LMIC       0.10              0.10               0.15   0.05           0.3           0.03
ALB             UMIC       0.20              0.15               0.20   0.08           0.4           0.04

=== spot check ===
  USA (HIC): recyc=0.35 sub=0.25 clinker=0.30 life=+50%
  DEU (HIC): recyc=0.35 sub=0.25 clinker=0.30 life=+50%
  CHN (UMIC): recyc=0.20 sub=0.15 clinker=0.20 life=+40%
  IND (LMIC): recyc=0.10 sub=0.10 clinker=0.15 life=+30%
  IDN (LMIC): recyc=0.10 sub=0.10 clinker=0.15 life=+30%
  NGA (LMIC): recyc=0.10 sub=

: 

## How the demand model consumes this

The MFA (`IMAGINE-Concrete`) CE scenario should read `CE-parameters-142.csv` and, for each economy, use its row's 2050 targets in the linear ramps of SI Eqs. S47 (intensive_use → ρ_use), S48 (lifetime_ext → μ_2050), S50 (reuse → ρ_reuse), S52 (recycling → ρ_recy), S54 (sand_substitution → ρ_sub,sand), S55 (clinker_ratio_target → ρ_cl), **replacing the former single global value**.

**Note — clinker differs from the other levers.** `clinker_ratio_target` is the *absolute* 2050 clinker-to-cement ratio (k-factor), not a 0→target fraction or a relative reduction. The MFA should linearly ramp each economy's **base-year** clinker ratio (GNR data) to this target: `k(t) = k_base + (target − k_base)·ramp(t)`; where a base ratio is already below its group target, the 2050 ratio is min(k_base, target) -- i.e. applied only as a reduction, so an economy already below its group target keeps its lower base ratio and is never forced upward.

The 'proportional knob' you already use still applies: a global CE-ambition multiplier `λ ∈ [0,1]` can scale every economy's target (`target_used = λ · target`), giving a one-dimensional CE-strength axis analogous to β in the allocation step, while preserving cross-country differentiation.